# Test the code from file `initialization.py`

# Importations

In [ ]:
# Numerical and scientific python programming
import numpy as np

import matplotlib.pyplot as plt

# Auxiliary python functions
import time

# Local imports
from moments.bloch import generate_pauli_basis, compute_tensor_basis, compute_subset_index_map
from moments.initialization import compute_param_from_X, compute_X_from_param, compute_initial_param_repeat

# Parametrization of the density matrix

First we test the fucntions that convert from parameter matrix to parameter vector and viceversa. These work either with standard density, or with Cholesky matrices.

- `compute_param_from_X`: It takes as input a numpy array representing a complex matrix `X`. Returns a numpy array representing a real vector `x` with real and imaginary values of `X`rearranged.
- `compute_X_from_param`: It takes as input a numpy array `x` of real values and appropiate dimension to parametriza a density matrix. It returns a numpy array with the entries of `x` rearanged representing a complex matrix `X`.

To check they work properly, we do two things:
1. Generate an initial matrix `X` -> convert to a parameter vector `x` -> reconstruct the matrix `X` -> compute the error.
2. generate an initial vector `x` -> convert to a matrix `X` -> reconstruct the parameter vector `x` -> compute the error.

Additionally, we also chech that the matrices and vectors generate fulfill the desired properties.

We do this for both the normal matrix `X` and the Cholesky parametrization.

## Non-Cholesky case

In [ ]:
# Define system dimenstion.
d = 5

# Initialize and reconstruct matrix.
X = np.random.randn(d, d) + 1j * np.random.randn(d, d)
x = compute_param_from_X(X)
X_rec = compute_X_from_param(x)

# Check vector properties.
print("Parameter vector is real:", np.isreal(x).all())
print("Parameter vector has appropriate dimension:", x.shape[0] == 2 * d**2)

# Check matrix properties.
print("\nParameter matrix is complex:", np.iscomplex(X).all())
print("Reconstructec parameter matrix is complex:", np.iscomplex(X_rec).all())
print("Reconstructec parameter matrix has appropriate dimension:", X_rec.shape == (d, d))

# Compute error.
print("\nOne way reconstruction works:", np.isclose(X, X_rec).all())

In [ ]:
# Initialize and reconstruct vector.
x = np.random.randn(2 * d**2)
X = compute_X_from_param(x)
x_rec = compute_param_from_X(X)

# Check vector properties.
print("Parameter vector is real:", np.isreal(x).all())
print("Reconstructed parameter vector is real:", np.isreal(x_rec).all())
print("Reconstructed parameter vector has appropriate dimension:", x_rec.shape[0] == 2 * d**2)

# Check matrix properties.
print("\nParameter matrix is complex:", np.iscomplex(X).all())
print("Parameter matrix has appropriate dimension:", X.shape == (d, d))

# Compute error.
print("\nOne way reconstruction works:", np.isclose(x, x_rec).all())

## Cholesky case

In [ ]:
# Define system dimention.
d = 5

# Initialize and reconstruct matrix.
X = np.random.randn(d, d) + 1j * np.random.randn(d, d)
X = np.tril(X)
x = compute_param_from_X(X, cholesky=True)
X_rec = compute_X_from_param(x, cholesky=True)

# Check vector properties.
print("Parameter vector is real:", np.isreal(x).all())
print("Parameter vector has appropriate dimension:", x.shape[0] == d * (d + 1))

# Chec matrix properties
print("\nParameter matrix is lower triangular:", np.allclose(X, np.tril(X)))
print("Reconstructed parameter matrix is lower triangular:", np.allclose(X_rec, np.tril(X_rec)))

print("\nParameter matrix is complex:", np.logical_or(np.iscomplex(X), np.isclose(X, np.zeros_like(X))).all())
print("Reconstructed parameter matrix is complex:", np.logical_or(np.iscomplex(X_rec), np.isclose(X_rec, np.zeros_like(X_rec))).all())
print("Reconstructed parameter matrix has appropriate dimension:", X_rec.shape == (d, d))

# Compute error
print("\nOne way reconstruction works:", np.isclose(X, X_rec).all())

In [ ]:
# Initialize and reconstruct vector.
x = np.random.randn(d * (d + 1))
X = compute_X_from_param(x, cholesky=True)
x_rec = compute_param_from_X(X, cholesky=True)

# Check vector properties
print("Parameter vector is real:", np.isreal(x).all())
print("Reconstructed parameter vector is real:", np.isreal(x_rec).all())
print("Reconstructed parameter vector has appropriate dimension:", x_rec.shape[0] == d * (d + 1))

# Check matrix properties
print("\nParameter matrix is complex:", np.logical_or(np.iscomplex(X), np.isclose(X, np.zeros_like(X))).all())
print("Parameter matrix has appropriate dimension:", X.shape == (d, d))

# Check error
print("\nOne way reconstruction works:", np.isclose(x, x_rec).all())

# Optimization functions

Now we test the behavour of the initialization function `compute_initial_param_repeat`.

- `compute_initial_param_repeat`: Finds a valid density matrix with the target Bloch lengths. It takes several inputs.
    - d: int. System dimention.
    - tensor_basis: np.ndarray. System operator basis.
    - subset_index_map: dict[tuple[int, ...], np.ndarray]. Relation between subsets $\mathbf M \subseteq \mathbf N$ of the set of subsytems and elements of the Bloch vector.
    - Rt: dict[tuple[int, ...], float]. Target Bloch lengths for every subset.
    - cholesky: bool = False. Whether or not we want to use the Cholesky parametrization
    - psd_tol: float = 1e-10. Tolerance for the positive semidefinite checks.
    - attempts: int = 5. Number of times the algorithm is run. The best result is kept.

We are interested in checking two things:
1. If the algorithm for initialization actually works.
2. If it works better with or whithout the Choleski parametrization.

## Space definition and discretization

The space of quantum states in terms of the Bloch lengts is a subset of $\mathbb R^3$, where each coordinate corresponds to one Bloch length and they satisfy several conditions. For more information we refer to Phys. Rev. A 109, 012423 (2024) or section 3.3 of the paper.

Denote $|\vec r_1| := x$, $|\vec r_2| := y$ and $|\vec r_{12}| := z$. The particular conditions for a two-qubit system read:

- $(x, y, z) \in [0, 1] \times [0, 1] \times [0, \sqrt3]$,
- $z \ge x + y - 1$,
- $z^2 \le 3 + x^2 + y^2 - 4xy - 4|x-y|$.

In [ ]:
# Define system parameters.
N = 2
d = 2 ** N

# Initialize the Pauli basis of a one-qubit system.
pauli_basis = generate_pauli_basis()
local_bases = [pauli_basis.copy()] * N
local_basis_sizes = [len(basis) for basis in local_bases]

# Compute tensor basis of the N qubit system.
tensor_basis = compute_tensor_basis(local_bases)
# Compute index mappings from basis elements to Bloch vector elements.
subset_index_map = compute_subset_index_map(local_basis_sizes)

In [ ]:
# Determine the number of grid points along each coordinate.
D = 20
Dx, Dy, Dz = D + 1, D + 1, int(np.sqrt(3) * D) + 1

# Discretize each coordinate.
x = np.linspace(0, 1, Dx)
y = np.linspace(0, 1, Dy)
z = np.linspace(0, np.sqrt(3), Dz)

# Construct the three-dimensional coordinate mesh.
X, Y, Z = np.meshgrid(x, y, z, indexing='ij')

# Define the physical-state constraints.
mask1 = Z >= X + Y - 1
mask2 = Z**2 <= 3 + X**2 + Y**2 - 4*X*Y - 4*np.abs(X - Y)
mask = mask1 & mask2

# Extract all physically allowed grid points for the loop.
points = np.column_stack((X[mask], Y[mask], Z[mask]))
total_points = points.shape[0]
print(f"Number of points: {total_points}")

## Optimization non-Cholesky

In [ ]:
# Data storage.
solutions, R_list, R_app = [], [], []
# Status storage.
times, success = [], []

# Iterate over valid points.
for counter, (x_val, y_val, z_val) in enumerate(points, 1):
    t0 = time.time()
    
    # Construct Boch length constraints.
    Rt = {(1,): float(x_val), (2,): float(y_val), (1, 2): float(z_val)}
    
    # Solve the maximization problem with a different strategy in each region.
    param_res = compute_initial_param_repeat(d, tensor_basis, subset_index_map, Rt)

    # Store results and data
    tf = time.time()
    times.append(tf - t0)
    R_app.append(param_res.moments)
    R_list.append(Rt)
    success.append(param_res.optimizer_info["success"])
    
    # Display status of the loop.
    percent_complete = (counter / total_points) * 100
    print(f"\rProgress: {percent_complete:.2f}% ({counter}/{total_points}) | Time for this point: {tf-t0:.3f} s", end="", flush=True)

print(f"\n\nDone!")
print(f"Successful optimizations: {sum(success)}/{total_points} | Percentage: {sum(success)*100/total_points:.2f} %")
print(f"Total time: {sum(times)/60:.2f} min | Average time per point: {sum(times)/total_points:.3f} s", )

In [ ]:
R_list = np.array(R_list)
R_app = np.array(R_app)

# Compute total time.
times = np.array(times)
print(f'Total time: {np.sum(times)/60:0.2f} min.\n')

# Compute errors.
errors = []
for xdx, R in enumerate(R_app):
    e = 0
    for subset in R.keys():
        e += np.abs(R[subset] - R_list[xdx][subset])
    errors.append(e)
errors = np.array(errors)

# Display errors.
print(f'Average error: {np.mean(errors)}.\n')

print(f'Max error: {errors[np.argmax(errors)]}.')
print(f'R_target = {R_list[np.argmax(errors)]}.')
print(f'R_app = {R_app[np.argmax(errors)]}.\n')

print(f'Min error: {errors[np.argmin(errors)]}.')
print(f'R_target = {R_list[np.argmin(errors)]}.')
print(f'R_app = {R_app[np.argmin(errors)]}.')

In [ ]:
# Display histogram of errors.
plt.hist(errors, bins=25)
plt.xlabel('Error')
plt.ylabel('Number of states')
plt.show()

## Optimization Cholesky

In [ ]:
# Data storage.
solutions, R_list, R_app = [], [], []
# Status storage.
times, success = [], []

# Iterate over valid points.
for counter, (x_val, y_val, z_val) in enumerate(points, 1):
    t0 = time.time()
    
    # Construct Boch length constraints.
    Rt = {(1,): float(x_val), (2,): float(y_val), (1, 2): float(z_val)}
    
    # Solve the maximization problem with a different strategy in each region.
    param_res = compute_initial_param_repeat(d, tensor_basis, subset_index_map, Rt, cholesky=True)

    # Store results and data
    tf = time.time()
    times.append(tf - t0)
    R_app.append(param_res.moments)
    R_list.append(Rt)
    success.append(param_res.optimizer_info["success"])
    
    # Display status of the loop.
    percent_complete = (counter / total_points) * 100
    print(f"\rProgress: {percent_complete:.2f}% ({counter}/{total_points}) | Time for this point: {tf-t0:.3f} s", end="", flush=True)

print(f"\n\nDone!")
print(f"Successful optimizations: {sum(success)}/{total_points} | Percentage: {sum(success)*100/total_points:.2f} %")
print(f"Total time: {sum(times)/60:.2f} min | Average time per point: {sum(times)/total_points:.3f} s", )

In [ ]:
R_list = np.array(R_list)
R_app = np.array(R_app)

# Compute total time.
times = np.array(times)
print(f'Total time: {np.sum(times)/60:0.2f} min.\n')

# Compute errors.
errors = []
for xdx, R in enumerate(R_app):
    e = 0
    for subset in R.keys():
        e += np.abs(R[subset] - R_list[xdx][subset])
    errors.append(e)
errors = np.array(errors)

# Display errors.
print(f'Average error: {np.mean(errors)}.\n')

print(f'Max error: {errors[np.argmax(errors)]}.')
print(f'R_target = {R_list[np.argmax(errors)]}.')
print(f'R_app = {R_app[np.argmax(errors)]}.\n')

print(f'Min error: {errors[np.argmin(errors)]}.')
print(f'R_target = {R_list[np.argmin(errors)]}.')
print(f'R_app = {R_app[np.argmin(errors)]}.')

In [ ]:
# Display histogram of errors.
plt.hist(errors, bins=25)
plt.xlabel('Error')
plt.ylabel('Number of states')
plt.show()